# Task 2 — EMT Recording Reader and Annotated Tracks

**Updated for your actual Hugging Face raw-cache layout and the shared `contracts.py` dictionary records.**

This notebook implements only Task 2: recording access, annotations, whole-recording splits, inspection, visualization, and tests. It does not train a detector or run tracking.

Your raw HF cache contains the required original recordings even though `load_dataset()` presents a flattened view. The reader uses:

- images: `images/<split>/images/<recording_id>/<frame>.jpg`;
- labels: `annotations/<split>/<recording_id>.txt`;
- splits: the existing `train/` and `test/` folders, with development recordings selected only from train;
- shared frame and observation dictionaries from `trackrag/contracts.py`.

**Pushable artifacts materialized by this notebook (repo-relative, outside the downloaded footage tree):**

- `trackrag/emt.py` — the reader;
- `tests/test_emt.py` — pytest file (Step 13);
- `splits/task2_manifest.json` — whole-recording split manifest (Step 7 / Step 12);
- `.gitignore` entry for the footage directory (Step 14).

The synthetic tests run without EMT. Real checks are enabled separately in Step 10.

## Step 1 — Your actual folder layout

On Vast.ai, `/workspace/EMT` links the two extracted HF cache locations together:

```text
/workspace/EMT/
├── images/
│   ├── train/
│   │   └── images/
│   │       ├── video_054604/
│   │       │   ├── 00001.jpg
│   │       │   ├── 00002.jpg
│   │       │   └── ...
│   │       └── video_220047/
│   └── test/
│       └── images/
└── annotations/
    ├── train/
    │   ├── video_054604.txt
    │   └── video_220047.txt
    └── test/
```

**Repo (what you push) stays separate from the footage:**

```text
<repo>/
├── trackrag/
│   ├── __init__.py
│   ├── contracts.py
│   └── emt.py
├── splits/
│   └── task2_manifest.json       # written here, committed to Git
├── tests/
│   └── test_emt.py               # written here, committed to Git
├── .gitignore                    # excludes /workspace/EMT and demo_emt
└── Task_2_EMT_Recording_Reader.ipynb
```

No `frames/`, `tracking/gmot/`, or `metadata.txt` is required. The existing test folder is treated as EMT's official test set. If OpenCV is missing, run `pip install opencv-python-headless matplotlib pytest` in a terminal.

In [ ]:
from pathlib import Path
import json
import random
import shutil

import cv2
import numpy as np
import matplotlib.pyplot as plt

Path("trackrag").mkdir(exist_ok=True)
Path("tests").mkdir(exist_ok=True)
Path("splits").mkdir(exist_ok=True)
Path("trackrag/__init__.py").write_text("")

# Use the project's shared contracts exactly as supplied; do not redefine them here.
contracts_path = Path("trackrag/contracts.py")
if not contracts_path.is_file():
    supplied = Path("/mnt/agents/upload/contracts.py")  # available in this workspace
    if supplied.is_file():
        shutil.copy2(supplied, contracts_path)
    else:
        raise FileNotFoundError(
            "Place the shared contracts.py at trackrag/contracts.py, then rerun this cell."
        )
print("Project package, tests/, splits/, and shared contracts ready.")

## Step 2 — Implement the split-first EMT reader

**Shared data contract**

- Frames and observations are ordinary dictionaries matching `trackrag/contracts.py`.
- Frame paths use the key `image_path`; boxes use `bbox_xyxy=(left,top,right,bottom)` in original pixels.
- Frame IDs come from numeric stems: `00001.jpg` becomes frame `1`.
- **Clock anchor:** the clock starts at **0 seconds on the first available image frame** (`first_frame_id`). At 10 FPS, frame 2 is 0.1 s: `timestamp_s=(frame_id-first_frame_id)/frame_rate`. In EMT the first image is normally also the first annotated frame, but the reader does not assume that — an annotation that starts later simply yields timestamps > 0.
- **Frame rate:** `open_recording()` uses `annotations/metadata.txt` if the release ships one (`read_release_frame_rate()`), otherwise it falls back to `DEFAULT_FRAME_RATE = 10.0`, documented as the assumed EMT release rate. Pass `frame_rate=` explicitly to override.
- `FrameNotFoundError` subclasses the contract's documented `KeyError`.
- Recordings open without labels. Only annotation loading raises `MissingLabelsError`.
- A recording ID found in both train and test is rejected rather than silently assigned twice.
- **Annotation column layout** (EMT GMOT-style, whitespace or comma separated): `frame_id track_id class_name <x> <y> <z> left top right bottom`. Columns `6..9` are original-pixel `xyxy`.
- Labels for unavailable image frames and `AV` ego-vehicle rows are skipped.
- `make_split_manifest()` writes to repo-relative `splits/task2_manifest.json` by default so the manifest can be committed; it is not written inside the downloaded footage tree.

In [ ]:
%%writefile trackrag/emt.py
"""Task 2 reader for EMT's split-first images/annotations layout."""
from __future__ import annotations
import argparse
import json
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator, Optional
import cv2
from trackrag.contracts import Frame, Recording, TrackObservation

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
DEFAULT_FRAME_RATE = 10.0
DEFAULT_MANIFEST_PATH = Path("splits") / "task2_manifest.json"
VEHICLE_CLASSES = {"Motorbike", "Small_motorised_vehicle", "Medium_vehicle", "Large_vehicle", "Car", "Bus", "Emergency_vehicle"}
SPLITS = ("train", "test")


class FrameNotFoundError(KeyError):
    """The requested original frame is absent."""


class MissingLabelsError(FileNotFoundError):
    """Labels were requested but the recording's label file is absent."""


@dataclass
class EMTRecording:
    recording_id: str
    data_root: Path
    split: str
    image_dir: Path
    frame_ids: list[int]
    frame_rate: float
    width: int
    height: int
    frame_index: dict[int, Path]

    @property
    def first_frame_id(self) -> int:
        return self.frame_ids[0]

    @property
    def annotation_path(self) -> Path:
        return self.data_root / "annotations" / self.split / f"{self.recording_id}.txt"

    def iter_frames(self) -> Iterator[Frame]:
        """Yield shared Frame dictionaries in ascending numerical frame order."""
        for frame_id in self.frame_ids:
            yield self.get_frame(frame_id)

    def get_frame(self, frame_id: int) -> Frame:
        """Return exactly the requested original frame; never substitute a neighbor.

        The recording clock is anchored at ``first_frame_id`` (the first available
        image frame). At ``frame_rate`` FPS, ``timestamp_s`` is
        ``(frame_id - first_frame_id) / frame_rate``. In EMT the first image frame
        is normally also the first annotated frame, but the reader does not assume
        that: an annotation that starts later simply yields timestamps > 0.
        """
        if frame_id not in self.frame_index:
            raise FrameNotFoundError(f"Frame {frame_id} is absent from recording {self.recording_id!r}.")
        path = self.frame_index[frame_id]
        image = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f"Cannot decode original frame image: {path}")
        height, width = image.shape[:2]
        return Frame(
            recording_id=self.recording_id,
            frame_id=frame_id,
            timestamp_s=(frame_id - self.first_frame_id) / self.frame_rate,
            width=width,
            height=height,
            image_path=str(path),
        )


def _candidate_image_dirs(data_root: Path, split: str, recording_id: str) -> list[Path]:
    base = data_root / "images" / split
    return [base / "images" / recording_id, base / recording_id]


def find_image_dir(data_root: Path, split: str, recording_id: str) -> Optional[Path]:
    return next((p for p in _candidate_image_dirs(data_root, split, recording_id) if p.is_dir()), None)


def _index_frames(image_dir: Path) -> dict[int, Path]:
    index: dict[int, Path] = {}
    for path in image_dir.iterdir():
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS or not path.stem.isdigit():
            continue
        frame_id = int(path.stem)
        if frame_id in index:
            raise ValueError(f"Duplicate frame {frame_id}: {index[frame_id].name} and {path.name}")
        index[frame_id] = path
    return index


def read_release_frame_rate(data_root: Path | str) -> Optional[float]:
    """Return the release's declared frame rate if ``annotations/metadata.txt`` exposes it.

    EMT publishes images and annotations separately, so a metadata file is *not*
    required. When it is absent this returns ``None`` and the caller falls back to
    ``DEFAULT_FRAME_RATE`` (documented as the assumed release rate).
    """
    path = Path(data_root) / "annotations" / "metadata.txt"
    if not path.is_file():
        return None
    for raw in path.read_text(encoding="utf-8-sig").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        for key in ("fps", "frame_rate", "framerate", "frame rate"):
            match = re.match(rf"^{re.escape(key)}\s*[:=]\s*([0-9.]+)$", line, re.IGNORECASE)
            if match:
                value = float(match.group(1))
                if value > 0:
                    return value
    return None


def open_recording(data_root: Path | str, recording_id: str, frame_rate: Optional[float] = None, split: Optional[str] = None) -> EMTRecording:
    """Open images/<split>/images/<recording_id>/ without requiring labels."""
    data_root = Path(data_root)
    if frame_rate is None:
        frame_rate = read_release_frame_rate(data_root) or DEFAULT_FRAME_RATE
    if frame_rate <= 0:
        raise ValueError("frame_rate must be positive")
    splits = [split] if split is not None else list(SPLITS)
    if any(s not in SPLITS for s in splits):
        raise ValueError("split must be train, test, or None")
    found = [(s, find_image_dir(data_root, s, recording_id)) for s in splits]
    found = [(s, p) for s, p in found if p is not None]
    if not found:
        raise FileNotFoundError(f"No image folder for {recording_id!r} under {data_root / 'images'}")
    if len(found) > 1:
        raise ValueError(f"Recording {recording_id!r} occurs in multiple splits: {[s for s, _ in found]}")
    recording_split, image_dir = found[0]
    frame_index = _index_frames(image_dir)
    if not frame_index:
        raise ValueError(f"Recording {recording_id!r} has no numerically named images")
    frame_ids = sorted(frame_index)
    first = cv2.imread(str(frame_index[frame_ids[0]]), cv2.IMREAD_COLOR)
    if first is None:
        raise ValueError(f"Cannot decode {frame_index[frame_ids[0]]}")
    height, width = first.shape[:2]
    return EMTRecording(recording_id, data_root, recording_split, image_dir, frame_ids, float(frame_rate), width, height, frame_index)


def load_annotated_tracks(recording: Recording, annotation_path: Optional[Path | str] = None) -> list[TrackObservation]:
    """Load original GMOT-style xyxy boxes as shared TrackObservation dictionaries.

    Column layout assumed by the EMT release (whitespace- or comma-separated):

        frame_id  track_id  class_name  <x>  <y>  <z>  left  top  right  bottom

    Columns ``6..9`` are original-pixel ``left, top, right, bottom`` (xyxy). Rows
    whose ``class_name`` is ``AV`` (ego vehicle) or whose ``frame_id`` has no
    matching image are skipped. A missing label file raises ``MissingLabelsError``
    so the recording can still be opened in images-only mode.
    """
    split = getattr(recording, "split")
    default = recording.data_root / "annotations" / split / f"{recording.recording_id}.txt"
    path = Path(annotation_path) if annotation_path is not None else default
    if not path.is_file() and annotation_path is None:
        alternate = default.with_suffix("")
        if alternate.is_file():
            path = alternate
    if not path.is_file():
        raise MissingLabelsError(f"No labels for {recording.recording_id!r}: {path}. Images-only mode remains available.")
    known_frames = set(recording.frame_ids)
    observations: list[TrackObservation] = []
    with path.open("r", encoding="utf-8-sig") as handle:
        for line_number, raw in enumerate(handle, 1):
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            parts = re.split(r"[,\s]+", line)
            try:
                frame_id = int(float(parts[0])); track_id = int(float(parts[1])); class_name = parts[2]
                left, top, right, bottom = map(float, parts[6:10])
            except (IndexError, ValueError) as exc:
                raise ValueError(f"{path}:{line_number}: invalid GMOT row: {raw!r}") from exc
            if class_name == "AV" or frame_id not in known_frames:
                continue
            if right <= left or bottom <= top:
                raise ValueError(f"{path}:{line_number}: box must have positive width and height")
            observations.append(TrackObservation(
                recording_id=recording.recording_id, frame_id=frame_id, track_id=track_id,
                class_name=class_name, bbox_xyxy=(left, top, right, bottom), source="annotated"))
    return sorted(observations, key=lambda o: (o["frame_id"], o["track_id"]))


def recording_ids_for_split(data_root: Path | str, split: str) -> list[str]:
    """List whole recordings represented by image folders in train or test."""
    if split not in SPLITS:
        raise ValueError("split must be train or test")
    base = Path(data_root) / "images" / split
    container = base / "images" if (base / "images").is_dir() else base
    if not container.is_dir():
        return []
    return sorted(p.name for p in container.iterdir() if p.is_dir() and _index_frames(p))


def assert_splits_disjoint(manifest: dict[str, list[str]], available_recording_ids: Optional[set[str]] = None) -> None:
    """Require exactly train/dev/test keys and no duplicate recording membership."""
    if set(manifest) != {"train", "dev", "test"}:
        raise ValueError("Manifest keys must be exactly train, dev, and test")
    owner: dict[str, str] = {}
    for split, ids in manifest.items():
        for recording_id in ids:
            if recording_id in owner:
                raise ValueError(f"Recording {recording_id!r} occurs in {owner[recording_id]} and {split}")
            owner[recording_id] = split
    if available_recording_ids is not None:
        listed = set(owner)
        missing = sorted(available_recording_ids - listed); extra = sorted(listed - available_recording_ids)
        if missing or extra:
            raise ValueError(f"Manifest/disk mismatch; missing={missing}, extra={extra}")


def make_split_manifest(data_root: Path | str, dev_count: int = 2, seed: int = 42, output_path: Optional[Path | str] = None) -> dict[str, list[str]]:
    """Keep every official test recording unchanged; select dev only from train.

    The manifest is written *outside* the downloaded footage tree by default so it
    can be committed: ``splits/task2_manifest.json`` relative to the repository.
    """
    if dev_count < 0:
        raise ValueError("dev_count must be non-negative")
    data_root = Path(data_root)
    official_train = recording_ids_for_split(data_root, "train")
    official_test = recording_ids_for_split(data_root, "test")
    overlap = set(official_train) & set(official_test)
    if overlap:
        raise ValueError(f"Recordings occur in both official train and test: {sorted(overlap)}")
    count = min(dev_count, max(0, len(official_train) - 1))
    chosen = set(random.Random(seed).sample(official_train, count)) if count else set()
    manifest = {"train": [r for r in official_train if r not in chosen], "dev": sorted(chosen), "test": official_test}
    assert_splits_disjoint(manifest, set(official_train) | set(official_test))
    output_path = Path(output_path) if output_path is not None else DEFAULT_MANIFEST_PATH
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(manifest, indent=2) + "\n")
    return manifest


def inspect_recording(data_root: Path | str, recording_id: str) -> None:
    """Print first/last frame details and one labeled object's original box."""
    recording = open_recording(data_root, recording_id)
    first = recording.get_frame(recording.frame_ids[0]); last = recording.get_frame(recording.frame_ids[-1])
    for label, frame in (("FIRST", first), ("LAST", last)):
        print(f"{label}: recording={frame['recording_id']} split={recording.split} frame={frame['frame_id']} timestamp_s={frame['timestamp_s']:.3f} dimensions={frame['width']}x{frame['height']} image_path={frame['image_path']}")
    try:
        observations = load_annotated_tracks(recording)
    except MissingLabelsError as exc:
        print(f"IMAGES ONLY: {exc}"); return
    if not observations:
        print("LABELS: no target observations reference available frames"); return
    chosen = next((o for o in observations if o["class_name"] in VEHICLE_CLASSES), observations[0])
    print(f"LABELED OBJECT: recording={chosen['recording_id']} frame={chosen['frame_id']} track={chosen['track_id']} class={chosen['class_name']} bbox_xyxy={chosen['bbox_xyxy']} source={chosen['source']}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=inspect_recording.__doc__)
    parser.add_argument("--data-root", required=True, type=Path); parser.add_argument("--recording-id", required=True)
    args = parser.parse_args(); inspect_recording(args.data_root, args.recording_id)


## Step 3 — Create a small split-first EMT fixture

The fixture uses your real layout, including the extra `images/<split>/images/` level. `rec_a` is created out of order as frames **3, 1, 5, 2**; frame **4 is missing**, and frame **3 has zero labels**. `rec_a` and `rec_b` share track ID **7** to verify recording-local identity.

In [ ]:
DATA_ROOT = Path("demo_emt")
for split in ("train", "test"):
    (DATA_ROOT / "images" / split / "images").mkdir(parents=True, exist_ok=True)
    (DATA_ROOT / "annotations" / split).mkdir(parents=True, exist_ok=True)
for recording_id in ["rec_a", "rec_b", "rec_c", "rec_images_only"]:
    (DATA_ROOT / "images" / "train" / "images" / recording_id).mkdir(parents=True, exist_ok=True)
(DATA_ROOT / "images" / "test" / "images" / "rec_official_test").mkdir(parents=True, exist_ok=True)

def write_test_frame(recording_id, frame_id, split="train", color=(30, 60, 90)):
    image = np.zeros((90, 160, 3), dtype=np.uint8); image[:] = color
    cv2.putText(image, f"{recording_id} f{frame_id}", (5, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 255, 255), 1)
    path = DATA_ROOT / "images" / split / "images" / recording_id / f"{frame_id:05d}.jpg"
    cv2.imwrite(str(path), image)
for frame_id in [3, 1, 5, 2]:
    write_test_frame("rec_a", frame_id)
write_test_frame("rec_b", 1, color=(40, 90, 50)); write_test_frame("rec_c", 1)
write_test_frame("rec_images_only", 1); write_test_frame("rec_official_test", 1, split="test")
(DATA_ROOT / "annotations" / "train" / "rec_a.txt").write_text(
    "00001 7 Car 0.0 0 0.0 20.0 15.0 70.0 70.0\n"
    "00002 7 Car 0.0 0 0.0 24.0 16.0 74.0 71.0\n"
    "00004 99 Car 0.0 0 0.0 10.0 10.0 30.0 30.0\n"
    "00005 8 Pedestrian 0.0 0 0.0 90.0 25.0 120.0 85.0\n"
    "00001 50 AV 0.0 0 0.0 0.0 0.0 10.0 10.0\n")
(DATA_ROOT / "annotations" / "train" / "rec_b.txt").write_text(
    "00001 7 Bus 0.0 0 0.0 30.0 20.0 100.0 80.0\n")
print("Split-first synthetic EMT fixture created under", DATA_ROOT.resolve())

## Step 4 — Test numerical order, exact lookup, dimensions, paths, and timestamps

The assertions below prove that iteration is numerical rather than filename-creation order, lookup returns the exact original path, and frame 5 at 10 FPS has timestamp **0.4 s** because the clock starts at frame 1.

In [ ]:
from trackrag.emt import open_recording, load_annotated_tracks, FrameNotFoundError, MissingLabelsError, inspect_recording

recording_a = open_recording(DATA_ROOT, "rec_a", frame_rate=10.0)
assert recording_a.split == "train"
assert recording_a.recording_id == "rec_a"
iterated = list(recording_a.iter_frames())
assert [f["frame_id"] for f in iterated] == [1, 2, 3, 5]
assert recording_a.frame_ids == [1, 2, 3, 5]
frame_5 = recording_a.get_frame(5)
assert isinstance(frame_5, dict)
assert set(frame_5) == {"recording_id", "frame_id", "timestamp_s", "width", "height", "image_path"}
assert frame_5["frame_id"] == 5
assert frame_5["image_path"].endswith("00005.jpg")
assert (frame_5["width"], frame_5["height"]) == (160, 90)
assert abs(frame_5["timestamp_s"] - 0.4) < 1e-12
assert iterated[0]["timestamp_s"] == 0.0
print("PASS: shared Frame dictionaries, numerical order, exact paths, dimensions, and 10 FPS timestamps")
print("Frame IDs:", recording_a.frame_ids)
print("Frame 5:", frame_5)

## Step 5 — Test missing frames, original annotation boxes, and zero-label frames

The missing frame must raise `FrameNotFoundError`; it must not silently return frame 3 or 5. Frame 3 exists on the timeline but has no annotations. The label for absent frame 4 is skipped.

In [ ]:
try:
    recording_a.get_frame(4)
    raise AssertionError("Missing frame 4 did not raise FrameNotFoundError")
except FrameNotFoundError as exc:
    print("PASS: documented KeyError-compatible missing-frame error:", exc)
observations_a = load_annotated_tracks(recording_a)
assert len(observations_a) == 3
assert [o["frame_id"] for o in observations_a] == [1, 2, 5]
assert observations_a[0]["bbox_xyxy"] == (20.0, 15.0, 70.0, 70.0)
assert all(isinstance(o, dict) for o in observations_a)
assert all(o["source"] == "annotated" for o in observations_a)
assert all(o["recording_id"] == "rec_a" for o in observations_a)
assert [o for o in observations_a if o["frame_id"] == 3] == []
assert [o for o in observations_a if o["class_name"] == "AV"] == []
print("PASS: shared observation dictionaries, original bbox_xyxy, zero-label frame 3, AV skipped")
for observation in observations_a:
    print(observation)

## Step 6 — Test images-only mode and recording-local track IDs

`rec_images_only` has no label file but still opens and decodes images. Track **7** exists in both `rec_a` and `rec_b`; loading `rec_b` must return only `rec_b`'s bus.

In [ ]:
images_only = open_recording(DATA_ROOT, "rec_images_only")
assert cv2.imread(images_only.get_frame(1)["image_path"]).shape == (90, 160, 3)
try:
    load_annotated_tracks(images_only)
    raise AssertionError("Missing labels did not raise MissingLabelsError")
except MissingLabelsError as exc:
    print("PASS: images-only recording opens; labels fail separately:", exc)
recording_b = open_recording(DATA_ROOT, "rec_b")
observations_b = load_annotated_tracks(recording_b)
assert observations_a[0]["track_id"] == observations_b[0]["track_id"] == 7
assert len(observations_b) == 1
assert observations_b[0]["recording_id"] == "rec_b"
assert observations_b[0]["class_name"] == "Bus"
assert all(o["recording_id"] == "rec_b" for o in observations_b)
print("PASS: shared numeric track IDs remain recording-local")
print("rec_a track 7:", observations_a[0]); print("rec_b track 7:", observations_b[0])

## Step 7 — Create and test the whole-recording split manifest

Your existing `images/train/` and `images/test/` folders define the official assignments. The helper selects development recordings **only from train**, leaves every test recording unchanged, and checks every available recording against exactly one final split. No `metadata.txt` is needed.

**The manifest is now written to repo-relative `splits/task2_manifest.json`** (outside the footage tree), so it can be committed and pushed.

In [ ]:
from trackrag.emt import recording_ids_for_split, make_split_manifest, assert_splits_disjoint, DEFAULT_MANIFEST_PATH
official_train = recording_ids_for_split(DATA_ROOT, "train")
official_test = recording_ids_for_split(DATA_ROOT, "test")
manifest = make_split_manifest(DATA_ROOT, dev_count=1, seed=42)
available = set(official_train) | set(official_test)
assert_splits_disjoint(manifest, available)
assert manifest["test"] == official_test == ["rec_official_test"]
assert set(manifest["train"] + manifest["dev"]) == set(official_train)
print("PASS: whole recordings are disjoint; official test is unchanged")
print(json.dumps(manifest, indent=2)); print("Saved to", DEFAULT_MANIFEST_PATH.resolve())
try:
    assert_splits_disjoint({"train": ["rec_a"], "dev": ["rec_a"], "test": []})
    raise AssertionError("Duplicate split membership was not detected")
except ValueError as exc:
    print("PASS: duplicate-split detection:", exc)

## Step 8 — Run the teammate inspection command

This command prints the first and last frame, timestamps, dimensions, local paths, and one labeled object's original box. For real EMT, replace the two arguments with your dataset path and recording ID.

In [ ]:
!python -m trackrag.emt --data-root demo_emt --recording-id rec_a

## Step 9 — Draw one annotated object

The helper opens the exact original frame and draws the unmodified annotated pixel box. This is the same helper used for the two real-recording visual checks below.

In [ ]:
def show_annotated_observation(recording, observation, ax=None):
    """Draw one shared TrackObservation on its exact original frame."""
    assert observation["recording_id"] == recording.recording_id
    frame = recording.get_frame(observation["frame_id"])
    canvas = cv2.imread(frame["image_path"], cv2.IMREAD_COLOR)
    if canvas is None:
        raise ValueError(f"Cannot decode {frame['image_path']}")
    x1, y1, x2, y2 = (round(v) for v in observation["bbox_xyxy"])
    cv2.rectangle(canvas, (x1, y1), (x2, y2), (0, 255, 0), 2)
    label = f"#{observation['track_id']} {observation['class_name']}"
    cv2.putText(canvas, label, (x1, max(15, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{recording.recording_id} — frame {frame['frame_id']} — {label}"); ax.axis("off")
    return ax
show_annotated_observation(recording_a, observations_a[0]); plt.show()
print("VISUAL CHECK: the green box should surround the annotated object on the original image.")

## Step 10 — Configure the real HF-cache EMT checks

The symlinked `/workspace/EMT` now points directly to your extracted HF images and annotations. Set `RUN_REAL_EMT_CHECKS=True` after creating those links.

**Frame rate is now sourced from the release** via `read_release_frame_rate()`, which reads `annotations/metadata.txt` if present. If the release does not ship one, the reader falls back to the documented `DEFAULT_FRAME_RATE = 10.0` and prints a clear warning so the timestamp assumption is explicit.

In [ ]:
from trackrag.emt import DEFAULT_FRAME_RATE, read_release_frame_rate

REAL_DATA_ROOT = Path("/workspace/EMT")
REAL_RECORDING_IDS = ["video_054604", "video_220047"]
RUN_REAL_EMT_CHECKS = False  # Set True only after original frames + annotations are available.

if not RUN_REAL_EMT_CHECKS:
    print("Real EMT checks skipped. Synthetic tests above remain valid.")
elif not REAL_DATA_ROOT.is_dir():
    raise FileNotFoundError(f"REAL-DATA BLOCKER: dataset root not found: {REAL_DATA_ROOT}")
else:
    detected = read_release_frame_rate(REAL_DATA_ROOT)
    if detected is None:
        print(f"No annotations/metadata.txt found; assuming EMT release frame rate = {DEFAULT_FRAME_RATE} FPS.")
        print("Timestamp assertions below are only valid if this assumption matches the pinned release.")
    else:
        print(f"Detected release frame rate from annotations/metadata.txt: {detected} FPS")

## Step 11 — Inspect two real recordings and visually verify labeled vehicles

For each recording, the code selects a labeled vehicle, opens its exact original frame, and displays the original box. **Manually confirm that each green rectangle surrounds the intended vehicle.** The inspection output also provides the first and last frame for a new teammate.

**Each chosen recording is asserted to be in the training split**, as the task requires ("Choose a training recording from the pinned EMT release"). If the assert fails, pick a different recording ID from `images/train/images/`.

In [ ]:
if RUN_REAL_EMT_CHECKS:
    real_recordings = []; real_vehicle_observations = []
    for recording_id in REAL_RECORDING_IDS:
        print("\n" + "=" * 30, recording_id, "=" * 30)
        inspect_recording(REAL_DATA_ROOT, recording_id)
        real = open_recording(REAL_DATA_ROOT, recording_id)
        assert real.split == "train", (
            f"{recording_id!r} is in split {real.split!r}, not 'train'; the task requires a training recording."
        )
        real_observations = load_annotated_tracks(real)
        vehicle = next((o for o in real_observations if o["class_name"] in {
            "Motorbike", "Small_motorised_vehicle", "Medium_vehicle", "Large_vehicle", "Car", "Bus", "Emergency_vehicle"}), None)
        if vehicle is None:
            raise ValueError(f"No labeled vehicle found in {recording_id}")
        real_recordings.append(real); real_vehicle_observations.append(vehicle)
    fig, axes = plt.subplots(1, len(real_recordings), figsize=(16, 6))
    if len(real_recordings) == 1:
        axes = [axes]
    for real, vehicle, ax in zip(real_recordings, real_vehicle_observations, axes):
        show_annotated_observation(real, vehicle, ax)
    plt.tight_layout(); plt.show()
    print("MANUAL REAL-DATA CHECK: verify the vehicle box in each displayed original frame.")
else:
    print("Skipped: enable RUN_REAL_EMT_CHECKS in Step 10.")

## Step 12 — Build the real recording-level manifest

This reads the official test IDs from the `images/test/` folders (the folder layout that Step 1 documents — **no `metadata.txt` is required or read here**), selects development recordings only from official train, checks every available frame folder against the manifest, and writes `splits/task2_manifest.json` **in the repository**, not inside the footage tree. It never creates a random frame-level split.

In [ ]:
if RUN_REAL_EMT_CHECKS:
    real_manifest = make_split_manifest(REAL_DATA_ROOT, dev_count=2, seed=42)
    real_available = set(recording_ids_for_split(REAL_DATA_ROOT, "train")) | set(recording_ids_for_split(REAL_DATA_ROOT, "test"))
    assert_splits_disjoint(real_manifest, real_available)
    assert real_manifest["test"] == recording_ids_for_split(REAL_DATA_ROOT, "test")
    print("PASS: every available recording is in exactly one split")
    print("PASS: all official EMT test recordings are unchanged")
    print({split: len(ids) for split, ids in real_manifest.items()})
    print("Manifest:", DEFAULT_MANIFEST_PATH.resolve())
else:
    print("Skipped: enable RUN_REAL_EMT_CHECKS in Step 10.")

## Step 13 — Write and run `tests/test_emt.py`

The task explicitly requires a pushable `tests/test_emt.py`. The cell below writes it with `%%writefile` (so it exists as a real file on disk) and then runs it with pytest. Every test that was previously inline is ported here, plus the new frame-rate checks.

In [ ]:
%%writefile tests/test_emt.py
"""Task 2 tests for the EMT recording reader.

Run from the repository root:

    pytest tests/test_emt.py -v
"""
from pathlib import Path

import cv2
import numpy as np
import pytest

from trackrag.emt import (
    DEFAULT_FRAME_RATE,
    FrameNotFoundError,
    MissingLabelsError,
    assert_splits_disjoint,
    load_annotated_tracks,
    make_split_manifest,
    open_recording,
    read_release_frame_rate,
    recording_ids_for_split,
)


def _write_frame(root: Path, recording_id: str, frame_id: int, split: str = "train", color=(30, 60, 90)) -> None:
    path = root / "images" / split / "images" / recording_id / f"{frame_id:05d}.jpg"
    path.parent.mkdir(parents=True, exist_ok=True)
    image = np.zeros((90, 160, 3), dtype=np.uint8)
    image[:] = color
    cv2.imwrite(str(path), image)


@pytest.fixture()
def emt_root(tmp_path: Path) -> Path:
    root = tmp_path / "emt"
    for split in ("train", "test"):
        (root / "images" / split / "images").mkdir(parents=True, exist_ok=True)
        (root / "annotations" / split).mkdir(parents=True, exist_ok=True)
    for frame_id in (3, 1, 5, 2):
        _write_frame(root, "rec_a", frame_id)
    _write_frame(root, "rec_b", 1, color=(40, 90, 50))
    _write_frame(root, "rec_images_only", 1)
    _write_frame(root, "rec_official_test", 1, split="test")
    (root / "annotations" / "train" / "rec_a.txt").write_text(
        "00001 7 Car 0.0 0 0.0 20.0 15.0 70.0 70.0\n"
        "00002 7 Car 0.0 0 0.0 24.0 16.0 74.0 71.0\n"
        "00004 99 Car 0.0 0 0.0 10.0 10.0 30.0 30.0\n"
        "00005 8 Pedestrian 0.0 0 0.0 90.0 25.0 120.0 85.0\n"
        "00001 50 AV 0.0 0 0.0 0.0 0.0 10.0 10.0\n"
    )
    (root / "annotations" / "train" / "rec_b.txt").write_text(
        "00001 7 Bus 0.0 0 0.0 30.0 20.0 100.0 80.0\n"
    )
    return root


def test_iter_frames_numerical_order(emt_root):
    rec = open_recording(emt_root, "rec_a", frame_rate=10.0)
    assert rec.split == "train"
    assert rec.frame_ids == [1, 2, 3, 5]
    assert [f["frame_id"] for f in rec.iter_frames()] == [1, 2, 3, 5]


def test_frame_contract_and_exact_path(emt_root):
    rec = open_recording(emt_root, "rec_a", frame_rate=10.0)
    frame = rec.get_frame(5)
    assert set(frame) == {"recording_id", "frame_id", "timestamp_s", "width", "height", "image_path"}
    assert frame["image_path"].endswith("00005.jpg")
    assert (frame["width"], frame["height"]) == (160, 90)
    assert abs(frame["timestamp_s"] - 0.4) < 1e-12


def test_clock_starts_at_first_image(emt_root):
    rec = open_recording(emt_root, "rec_a", frame_rate=10.0)
    assert rec.get_frame(rec.first_frame_id)["timestamp_s"] == 0.0


def test_missing_frame_raises_keyerror(emt_root):
    rec = open_recording(emt_root, "rec_a")
    with pytest.raises(FrameNotFoundError):
        rec.get_frame(4)
    with pytest.raises(KeyError):
        rec.get_frame(4)


def test_annotations_use_original_boxes(emt_root):
    rec = open_recording(emt_root, "rec_a")
    obs = load_annotated_tracks(rec)
    assert [o["frame_id"] for o in obs] == [1, 2, 5]
    assert obs[0]["bbox_xyxy"] == (20.0, 15.0, 70.0, 70.0)
    assert all(o["source"] == "annotated" for o in obs)
    assert all(o["recording_id"] == "rec_a" for o in obs)
    assert [o for o in obs if o["frame_id"] == 3] == []
    assert [o for o in obs if o["class_name"] == "AV"] == []


def test_images_only_mode(emt_root):
    rec = open_recording(emt_root, "rec_images_only")
    assert cv2.imread(rec.get_frame(1)["image_path"]).shape == (90, 160, 3)
    with pytest.raises(MissingLabelsError):
        load_annotated_tracks(rec)


def test_track_ids_are_recording_local(emt_root):
    obs_a = load_annotated_tracks(open_recording(emt_root, "rec_a"))
    obs_b = load_annotated_tracks(open_recording(emt_root, "rec_b"))
    assert obs_a[0]["track_id"] == obs_b[0]["track_id"] == 7
    assert {o["recording_id"] for o in obs_b} == {"rec_b"}
    assert obs_b[0]["class_name"] == "Bus"


def test_manifest_is_disjoint_and_preserves_official_test(emt_root, tmp_path):
    manifest = make_split_manifest(emt_root, dev_count=1, seed=42, output_path=tmp_path / "manifest.json")
    available = set(recording_ids_for_split(emt_root, "train")) | set(recording_ids_for_split(emt_root, "test"))
    assert_splits_disjoint(manifest, available)
    assert manifest["test"] == ["rec_official_test"]
    assert set(manifest["train"]) | set(manifest["dev"]) | set(manifest["test"]) == available
    assert set(manifest["train"]) | set(manifest["dev"]) == set(recording_ids_for_split(emt_root, "train"))


def test_duplicate_split_detected():
    with pytest.raises(ValueError):
        assert_splits_disjoint({"train": ["x"], "dev": ["x"], "test": []})


def test_frame_rate_validation(emt_root):
    with pytest.raises(ValueError):
        open_recording(emt_root, "rec_a", frame_rate=0)


def test_release_frame_rate_missing_returns_none(emt_root):
    assert read_release_frame_rate(emt_root) is None


def test_default_frame_rate_is_used_when_metadata_absent(emt_root):
    rec = open_recording(emt_root, "rec_a")
    assert rec.frame_rate == DEFAULT_FRAME_RATE


In [ ]:
!python -m pytest tests/test_emt.py -q

## Step 14 — `.gitignore` and the push checklist

The task says: *"Keep the downloaded footage outside Git."* and *"Push the reader, split manifest, local setup instructions, tests and the verification command."*

This cell writes a `.gitignore` that excludes the footage tree, then lists exactly what to commit.

In [ ]:
gitignore = Path(".gitignore")
existing = gitignore.read_text() if gitignore.is_file() else ""
entries = ["/workspace/EMT/", "demo_emt/", "__pycache__/", "*.pyc", ".pytest_cache/", "*.egg-info/"]
lines = [line for line in existing.splitlines() if line.strip()]
for entry in entries:
    if entry not in lines:
        lines.append(entry)
gitignore.write_text("\n".join(lines) + "\n")
print("Wrote .gitignore:"); print(gitignore.read_text())
print("\nFiles to push:")
for path in [
    Path("trackrag/emt.py"),
    Path("tests/test_emt.py"),
    Path("splits/task2_manifest.json"),
    Path(".gitignore"),
    Path("Task_2_EMT_Recording_Reader.ipynb"),
]:
    exists = "✓" if path.exists() else "✗ (run earlier cells first)"
    print(f"  {exists}  {path}")
print("\nVerification command for a teammate:")
print("  python -m trackrag.emt --data-root /workspace/EMT --recording-id video_054604")

## Completion checklist

- [x] `open_recording()` supports your exact HF raw-cache layout.
- [x] Frames and observations match the shared `contracts.py` dictionary fields.
- [x] `image_path` and `bbox_xyxy` are used for Tasks 1, 4, and 5 integration.
- [x] Missing frames raise a `KeyError`-compatible error; images-only mode works.
- [x] Clock anchor is documented (`first_frame_id`, first available image frame).
- [x] Annotation column layout is documented in `load_annotated_tracks`'s docstring.
- [x] Frame rate is sourced from `annotations/metadata.txt` if present, otherwise the documented `DEFAULT_FRAME_RATE` fallback is used and printed.
- [x] Whole-recording dev selection comes only from train; official test remains unchanged.
- [x] Real recordings are asserted to be in the `train` split.
- [x] `splits/task2_manifest.json` is repo-relative, outside the footage tree, and commit-ready.
- [x] `tests/test_emt.py` is written to disk and runnable with `pytest tests/test_emt.py -v`.
- [x] `.gitignore` excludes the footage tree.
- [ ] Visually confirm two real labeled vehicles after enabling Step 10 and running Steps 11–12 on GPU.

The reusable implementation is written to the expected integration path `trackrag/emt.py`; the real manifest is written to `splits/task2_manifest.json`; tests are written to `tests/test_emt.py`.